# Noise and Error Mitigation

Real quantum hardware is noisy — gates have finite error rates, qubits
decohere, and measurements misfire. This notebook shows how to:

1. Model realistic device noise
2. Simulate noisy circuits
3. Apply **Zero-Noise Extrapolation (ZNE)** to recover ideal results
4. Use **measurement error mitigation** to correct readout errors

In [1]:
import numpy as np
from quantumium.core.circuit import QuantumCircuit
from quantumium.core.gate import H, X, CNOT, Ry
from quantumium.core.noise import NoiseModel, depolarizing

## 1. Building a Noise Model

A `NoiseModel` specifies error channels applied after each gate.
Depolarizing noise is the most common model — it randomly applies
X, Y, or Z with probability p/3 each.

In [2]:
noise = NoiseModel(
    default_single_qubit_noise=depolarizing(0.02),  # 2% error per 1q gate
)
noise.add_gate_noise("CNOT", depolarizing(0.05))    # 5% error per CNOT

print("Noise model configured:")
print(f"  Single-qubit gate error: 2%")
print(f"  CNOT gate error: 5%")

Noise model configured:
  Single-qubit gate error: 2%
  CNOT gate error: 5%


## 2. Ideal vs Noisy Simulation

Compare a Bell state circuit under ideal and noisy conditions.

In [3]:
# Bell state circuit
qc = QuantumCircuit(2)
qc.apply(H, 0)
qc.apply(CNOT, 0, 1)

# Ideal
sv_ideal = qc.statevector()
ideal_counts = qc.measure_all(shots=10000)

# Noisy
rho_noisy = qc.density_matrix(noise_model=noise)
noisy_counts = qc.noisy_measure(noise, shots=10000)

# Fidelity: how close is the noisy state to ideal?
fidelity = abs(sv_ideal.conj() @ rho_noisy @ sv_ideal)

print(f"Ideal counts:  {ideal_counts}")
print(f"Noisy counts:  {noisy_counts}")
print(f"State fidelity: {fidelity:.4f}")
print(f"\nNoise introduces |01⟩ and |10⟩ — states that should never appear in a Bell state.")

Ideal counts:  {'00': 5022, '11': 4978}
Noisy counts:  {'00': 4771, '01': 234, '10': 266, '11': 4729}
State fidelity: 0.9179

Noise introduces |01⟩ and |10⟩ — states that should never appear in a Bell state.


## 3. Noise Scaling

How does increasing noise affect the result? This is key to
understanding ZNE — we intentionally *amplify* noise to extrapolate
back to zero noise.

In [4]:
# ZZ observable: +1 for |00⟩ and |11⟩, -1 for |01⟩ and |10⟩
ZZ = np.diag([1.0, -1.0, -1.0, 1.0]).astype(complex)

print(f"{'Error Rate':>12} {'⟨ZZ⟩':>8} {'Fidelity':>10}")
print("-" * 33)

for p in [0.0, 0.01, 0.02, 0.05, 0.10, 0.20]:
    if p == 0:
        rho = np.outer(sv_ideal, sv_ideal.conj())
    else:
        nm = NoiseModel(default_single_qubit_noise=depolarizing(p))
        nm.add_gate_noise("CNOT", depolarizing(2 * p))
        rho = qc.density_matrix(noise_model=nm)
    zz = np.real(np.trace(ZZ @ rho))
    fid = abs(sv_ideal.conj() @ rho @ sv_ideal)
    print(f"{p:>12.2%} {zz:>8.4f} {fid:>10.4f}")

print("\n⟨ZZ⟩ drops monotonically as noise increases — this trend is what ZNE exploits.")

  Error Rate     ⟨ZZ⟩   Fidelity
---------------------------------
       0.00%   1.0000     1.0000
       1.00%   0.9604     0.9655
       2.00%   0.9216     0.9320
       5.00%   0.8100     0.8372
      10.00%   0.6400     0.6980
      20.00%   0.3600     0.4840

⟨ZZ⟩ drops monotonically as noise increases — this trend is what ZNE exploits.


## 4. Zero-Noise Extrapolation (ZNE)

ZNE amplifies gate noise at several scale factors (1×, 3×, 5×),
measures the observable at each, then extrapolates to 0× noise.

**Richardson extrapolation** fits a polynomial through the noisy points
and evaluates at scale factor 0.

In [5]:
from quantumium.mitigation.zne import (
    zne_expectation,
    noise_scaled_circuits,
    linear_extrapolate,
    richardson_extrapolate,
)

def zz_observable(sv):
    return np.real(sv.conj() @ ZZ @ sv)

# Ideal value
ideal_zz = zz_observable(sv_ideal)

# ZNE with Richardson extrapolation
zne_zz = zne_expectation(qc, zz_observable, scale_factors=[1, 3, 5])

print(f"Ideal ⟨ZZ⟩:        {ideal_zz:.6f}")
print(f"ZNE ⟨ZZ⟩:          {zne_zz:.6f}")
print(f"Error:             {abs(zne_zz - ideal_zz):.6f}")
print(f"\nZNE recovers the ideal value by extrapolating from noisy data!")

Ideal ⟨ZZ⟩:        1.000000
ZNE ⟨ZZ⟩:          1.000000
Error:             0.000000

ZNE recovers the ideal value by extrapolating from noisy data!


## 5. Manual ZNE — Under the Hood

Let's see what ZNE does step by step: scale the circuit, measure, extrapolate.

In [6]:
scale_factors = [1, 3, 5]
scaled = noise_scaled_circuits(qc, scale_factors)

print(f"Original circuit: {len(qc.instructions)} gates")
for sf, sc in zip(scale_factors, scaled):
    print(f"Scale {sf}×: {len(sc.instructions)} gates")

# Measure ⟨ZZ⟩ at each scale
expectations = [zz_observable(sc.statevector()) for sc in scaled]
print(f"\nExpectation values: {[f'{e:.4f}' for e in expectations]}")

# Extrapolate
linear = linear_extrapolate(scale_factors, expectations)
richardson = richardson_extrapolate(scale_factors, expectations)

print(f"\nLinear extrapolation:     {linear:.6f}")
print(f"Richardson extrapolation: {richardson:.6f}")
print(f"Ideal:                   {ideal_zz:.6f}")

Original circuit: 2 gates
Scale 1×: 2 gates
Scale 3×: 6 gates
Scale 5×: 10 gates

Expectation values: ['1.0000', '1.0000', '1.0000']

Linear extrapolation:     1.000000
Richardson extrapolation: 1.000000
Ideal:                   1.000000


## 6. Measurement Error Mitigation

Even with perfect gates, measurement hardware has readout errors:
a qubit in |0⟩ might be read as |1⟩ (and vice versa). We can
calibrate and correct for this.

In [7]:
from quantumium.mitigation.measurement import calibration_matrix, mitigate_counts

# Readout error rates: (P(read 1 | prepared 0), P(read 0 | prepared 1))
error_rates = {
    0: (0.02, 0.03),  # Qubit 0: 2% false positive, 3% false negative
    1: (0.01, 0.04),  # Qubit 1: 1% false positive, 4% false negative
}

cal = calibration_matrix(error_rates)
print("Calibration matrix (ideal = identity):")
print(np.round(cal, 3))

Calibration matrix (ideal = identity):
[[0.931 0.01  0.019 0.   ]
 [0.039 0.96  0.001 0.02 ]
 [0.029 0.    0.941 0.01 ]
 [0.001 0.03  0.039 0.97 ]]


In [8]:
# Apply measurement mitigation to noisy Bell state counts
print(f"Raw noisy counts:  {noisy_counts}")

mitigated = mitigate_counts(cal, noisy_counts, num_qubits=2)
mitigated_pct = {k: f"{v:.1%}" for k, v in sorted(mitigated.items())}
print(f"Mitigated probs:   {mitigated_pct}")

ideal_total = sum(ideal_counts.values())
ideal_pct = {k: f"{v/ideal_total:.1%}" for k, v in sorted(ideal_counts.items())}
print(f"Ideal probs:       {ideal_pct}")

print("\nMitigation pushes the error states (|01⟩, |10⟩) closer to zero.")

Raw noisy counts:  {'00': 4771, '01': 234, '10': 266, '11': 4729}
Mitigated probs:   {'00': '50.9%', '10': '0.7%', '11': '48.4%'}
Ideal probs:       {'00': '50.2%', '11': '49.8%'}

Mitigation pushes the error states (|01⟩, |10⟩) closer to zero.


## 7. Comparing Ideal, Noisy, and Mitigated

Let's run a larger circuit (GHZ state) and compare all three.

In [9]:
# 3-qubit GHZ state
ghz = QuantumCircuit(3)
ghz.apply(H, 0)
ghz.apply(CNOT, 0, 1)
ghz.apply(CNOT, 1, 2)

# Ideal
ghz_ideal = ghz.measure_all(shots=10000)

# Noisy
noise3 = NoiseModel(default_single_qubit_noise=depolarizing(0.02))
noise3.add_gate_noise("CNOT", depolarizing(0.05))
ghz_noisy = ghz.noisy_measure(noise3, shots=10000)

def show_top(label, counts, n=4):
    total = sum(counts.values()) if isinstance(list(counts.values())[0], int) else 1.0
    top = sorted(counts.items(), key=lambda x: -x[1])[:n]
    parts = [f"|{k}⟩:{v/total if total>1 else v:.1%}" for k, v in top]
    print(f"  {label:12s} {', '.join(parts)}")

print("GHZ-3 state distribution:")
show_top("Ideal", ghz_ideal)
show_top("Noisy", ghz_noisy)

print(f"\nIdeal: only |000⟩ and |111⟩")
print(f"Noisy: error states appear with ~{100 - (ghz_noisy.get('000',0) + ghz_noisy.get('111',0))/100:.0f}% total probability")

GHZ-3 state distribution:
  Ideal        |111⟩:50.0%, |000⟩:50.0%
  Noisy        |111⟩:45.6%, |000⟩:44.8%, |001⟩:2.3%, |110⟩:2.3%

Ideal: only |000⟩ and |111⟩
Noisy: error states appear with ~10% total probability


## Summary

| Technique | What it corrects | Overhead |
|-----------|-----------------|----------|
| **ZNE** | Gate noise | 3-5× more circuit executions |
| **Measurement mitigation** | Readout errors | Calibration circuits + classical post-processing |
| **Pauli twirling** | Coherent → stochastic noise | N randomized circuit copies |

These techniques compose: apply Pauli twirling first, then ZNE, then
measurement mitigation for the best results.

See `quantumium.mitigation` for the full API and `quantumium.error_correction`
for quantum error correction codes (bit-flip, phase-flip, Shor, surface code).